<a href="https://colab.research.google.com/github/ltphongssvn/cs1090b_HallucinationLegalRAGChatbots/blob/feature%2Fdata-acquisition/Making_the_Most_of_your_Colab_Subscription.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS1090B — Hallucination in Legal RAG Chatbots — End-to-End Pipeline

Runs the data pipeline described in the README on Google Colab Pro A100.
Stages: bootstrap → env check → CourtListener ingest → data audit → dataset probe → LePaRD ingest → LePaRD↔CL compat audit.

In [ ]:
# Cell 2: Bootstrap repo + uv venv (pipeline cells will subprocess via .venv/bin/python)
import os
import pathlib
import subprocess

REPO_URL = "https://github.com/ltphongssvn/cs1090b_HallucinationLegalRAGChatbots.git"
REPO_DIR = "/content/cs1090b_HallucinationLegalRAGChatbots"

def sh(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True)
    if check and r.returncode != 0:
        raise SystemExit(f"command failed: {cmd}")


if not pathlib.Path(REPO_DIR).exists():
    sh(f"git clone {REPO_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

if subprocess.run("command -v uv", shell=True).returncode != 0:
    sh("curl -LsSf https://astral.sh/uv/install.sh | sh")
    os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"

sh("uv python install 3.11.9")
sh("uv python pin 3.11.9")
if not pathlib.Path(".venv").exists():
    sh("uv sync")

pathlib.Path(".env").write_text(
    "export PYTHONHASHSEED=0\n"
    "export CUBLAS_WORKSPACE_CONFIG=:4096:8\n"
    "export TOKENIZERS_PARALLELISM=false\n"
    "export RANDOM_SEED=0\n"
    "export TARGET_GPU_COUNT=1\n"
    "export TARGET_VRAM_GB_MIN=20.0\n"
)

# Verify .venv has correct pinned versions
r = subprocess.run(
    [
        ".venv/bin/python",
        "-c",
        "import sys, numpy, torch, transformers; "
        "print(f'python {sys.version.split()[0]}'); "
        "print(f'numpy {numpy.__version__}'); "
        "print(f'torch {torch.__version__}'); "
        "print(f'transformers {transformers.__version__}')",
    ],
    capture_output=True,
    text=True,
)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
    raise SystemExit("venv verification failed")

$ git clone https://github.com/ltphongssvn/cs1090b_HallucinationLegalRAGChatbots.git /content/cs1090b_HallucinationLegalRAGChatbots
cwd: /content/cs1090b_HallucinationLegalRAGChatbots
$ uv python install 3.11.9
$ uv python pin 3.11.9
$ uv sync
python 3.11.9
numpy 1.26.4
torch 2.0.1+cu117
transformers 4.41.2



# Login with a web browser, at Terminal prompt
git config --global user.email "ltphongssvn@gmail.com" && git config --global user.name "ltphongssvn"


gh auth login --hostname github.com --git-protocol https --web

In [4]:
# Cell 3: Environment verification — runs notebooks/cs1090b_HallucinationLegalRAGChatbots.md Cell 1
import os
import pathlib
import re
import subprocess
import sys

os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

nb_src = pathlib.Path("notebooks/cs1090b_HallucinationLegalRAGChatbots.md").read_text()
blocks = re.findall(r"```\{code-cell\}[^\n]*\n(.*?)```", nb_src, flags=re.DOTALL)
assert blocks, "no code cells found in notebook md"
cell1_code = blocks[0].replace("os.chdir(os.path.dirname(os.getcwd()))", "# chdir not needed — already at repo root")

env = {**os.environ, "PYTHONUNBUFFERED": "1"}
proc = subprocess.Popen(
    [".venv/bin/python", "-u", "-c", cell1_code],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    text=True,
    env=env,
)
for line in proc.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
proc.wait()
if proc.returncode != 0:
    raise SystemExit(f"Cell 3 failed with exit code {proc.returncode}")

  [repro] Reproducibility configured:
    PYTHONHASHSEED=0
    CUBLAS_WORKSPACE_CONFIG=:4096:8
    TOKENIZERS_PARALLELISM=false
    deterministic_algorithms=True
    cudnn_benchmark=False
    cudnn_deterministic=True
    random_seed=0
    torch.cuda.manual_seed_all(0) → 1 GPU(s)
    TDD RED→GREEN: Environment Contract
  ✓ PASS: Every required dependency must be importable and meet version constraints
  ✓ PASS: CUDA GPU must be detected for training
  ✓ PASS: GPU must have at least 10GB VRAM for transformer fine-tuning
  ✓ PASS: PyTorch must be compiled with CUDA support
  ✓ PASS: Cross-dependency version constraints must be satisfied
  
    Preflight Checks — Failure Isolation Gate
  ✓ PASS: GPU count 1 (allocated)
  ✓ PASS: GPU[0] NVIDIA A100-SXM4-80GB | cap (8, 0) | 85.1GB
  ✓ PASS: torch CUDA runtime 11.7
  ✓ PASS: Disk 201.9GB free
  ✓ PASS: src/repro.py importable
  ✓ PASS: repro_cfg['PYTHONHASHSEED'] = '0'
  ✓ PASS: repro_cfg['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
  ✓ PASS: repro

In [ ]:
# Cell 5: GCS Setup and Authentication
#
# Data flow (set up by this cell + Cell 5b):
#   S3 public bucket  ──►  Google Drive (persistent staging)  ──►  GCS
#                           ^ survives Colab runtime disconnects
#   GCS  ──►  all downstream probe / audit / train cells (streamed via gcsfs)
#
# This cell only authenticates and builds the configs; Cell 5b performs the
# actual Drive download + GCS upload (idempotent, skips anything already in
# GCS or already on Drive).
import os
import pathlib
import subprocess
import sys
import gcsfs

# Add repo to path for imports
sys.path.insert(0, "/content/cs1090b_HallucinationLegalRAGChatbots")

from src.gcs_utils import authenticate_gcs_colab, GCSConfig
from src.config import PipelineConfig
from google.colab import userdata

# Authenticate with GCS (Colab will prompt for authentication)
authenticate_gcs_colab()

# Load GCS credentials from Colab secrets
GCS_BUCKET_NAME = userdata.get("GCS_BUCKET_NAME")
GCS_PROJECT_ID = userdata.get("GCS_PROJECT_ID")

if GCS_BUCKET_NAME and GCS_PROJECT_ID:
    print(f"Loaded GCS credentials from Colab secrets")
else:
    raise RuntimeError(
        "GCS_BUCKET_NAME and GCS_PROJECT_ID must be set in Colab secrets. "
        "Add them via the key icon in the left sidebar."
    )

# Initialize configurations with explicit GCS settings
pipeline_cfg = PipelineConfig(
    use_gcs_streaming=True,
    gcs_bucket_name=GCS_BUCKET_NAME,
    gcs_project_id=GCS_PROJECT_ID,
    gcs_cl_bulk_prefix="cs1090b_cl_bulk/",
    gcs_cl_shards_prefix="cs1090b_cl_federal_appellate_bulk/",
    gcs_lepard_prefix="cs1090b_lepard/"
)

gcs_cfg = GCSConfig(
    bucket_name=GCS_BUCKET_NAME,
    project_id=GCS_PROJECT_ID,
    use_gcs_streaming=True
)

print(f"GCS bucket:  {gcs_cfg.bucket_name}")
print(f"GCS project: {gcs_cfg.project_id}")
print("\n✓ GCS streaming ready")
print("Cell 5b will ensure the datasets are in GCS (Drive-staged download →")
print("upload, idempotent). Downstream cells then stream directly from GCS.")


Mounted at /content/drive
Drive free: 191.8 GB / total: 259.7 GB
/content/drive/MyDrive/cs1090b_cl_bulk ready


In [ ]:
# Cell 5b: Ensure all datasets are in GCS (idempotent, Colab-resilient)
#
# Single entry point: `scripts/upload_data_to_gcs.sh`. For each dataset
# (shards, LePaRD, CourtListener bulk) the script skips any file already in
# GCS, otherwise downloads into the Drive-backed staging dir and uploads.
# Drive persistence → files survive Colab runtime disconnects; re-running
# this cell resumes from the last completed file.
import os
import subprocess
import sys

os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

# 1. Mount Drive (no-op if already mounted).
try:
    from google.colab import drive  # type: ignore[import-untyped]
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("Not running in Colab — staging will fall back to /tmp.")

# 2. Drive-backed staging dir for the CourtListener bulk CSVs.
STAGING_DIR = os.environ.get(
    "CL_BULK_STAGING_DIR",
    "/content/drive/MyDrive/cs1090b_cl_bulk_staging"
    if os.path.isdir("/content/drive/MyDrive")
    else "/tmp/courtlistener_bulk",
)
os.makedirs(STAGING_DIR, exist_ok=True)
print(f"Bucket:      gs://{GCS_BUCKET_NAME}/")
print(f"Staging dir: {STAGING_DIR}")
print()

# 3. Run the upload script with real-time output. The script is idempotent —
#    any dataset already in GCS is a fast skip, so this cell is cheap to
#    re-run after a disconnect.
env = {
    **os.environ,
    "GCS_BUCKET": GCS_BUCKET_NAME,
    "GCS_CL_BULK_PREFIX": "cs1090b_cl_bulk",
    "CL_BULK_STAGING_DIR": STAGING_DIR,
    "PYTHONUNBUFFERED": "1",
}
proc = subprocess.Popen(
    ["bash", "scripts/upload_data_to_gcs.sh"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    text=True,
    env=env,
)
for line in proc.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
proc.wait()

if proc.returncode != 0:
    raise RuntimeError(
        f"upload_data_to_gcs.sh exited with code {proc.returncode}. "
        "Re-run this cell to resume from the last completed file."
    )
print("\n✓ All datasets (shards, LePaRD, CourtListener bulk) present in GCS.")


In [ ]:
# Verify CourtListener bulk CSVs are present in GCS
import subprocess
import sys

os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

print("=" * 60)
print("  Verifying CourtListener bulk CSVs in GCS")
print("=" * 60)

# Use the verification script
proc = subprocess.run(
    [
        ".venv/bin/python", 
        "scripts/verify_gcs_data.py",
        "--bucket", GCS_BUCKET_NAME,
        "--project", GCS_PROJECT_ID,
        "--check-bulk",
        "--no-auth"
    ],
    capture_output=True,
    text=True
)

print(proc.stdout)
if proc.stderr:
    print(proc.stderr, file=sys.stderr)

if proc.returncode != 0:
    raise RuntimeError("Required bulk CSV files not found in GCS. Please upload them first.")

print(f"\n✓ All bulk CSVs verified in GCS")

already symlinked -> /content/drive/MyDrive/cs1090b_cl_bulk
  courts-2026-03-31.csv.bz2  0.00 GB
  dockets-2026-03-31.csv.bz2  4.98 GB
  opinion-clusters-2026-03-31.csv.bz2  2.45 GB
  opinions-2026-03-31.csv.bz2  54.19 GB


In [ ]:
# Placeholder — bulk CSVs are staged to Drive then uploaded to GCS by Cell 5b.
# Downstream cells stream from GCS, so no further local download is needed.
print("✓ Bulk CSV acquisition handled by Cell 5b (Drive-staged → GCS).")
print(f"  Pipeline streams from: gs://{GCS_BUCKET_NAME}/{pipeline_cfg.gcs_cl_bulk_prefix}")


bulk_dir -> /content/drive/MyDrive/cs1090b_cl_bulk
  All 4 bulk CSVs already present in data/raw/cl_bulk - skipping
    courts-2026-03-31.csv.bz2  0.00 GB
    dockets-2026-03-31.csv.bz2  4.98 GB
    opinion-clusters-2026-03-31.csv.bz2  2.45 GB
    opinions-2026-03-31.csv.bz2  54.19 GB


In [ ]:
# Cell 5: Verify shards are present in GCS (skip pipeline if already processed)
import os
import subprocess
import sys

os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

print("=" * 60)
print("  Verifying processed shards in GCS")
print("=" * 60)

# Use the verification script
proc = subprocess.run(
    [
        ".venv/bin/python",
        "scripts/verify_gcs_data.py",
        "--bucket", GCS_BUCKET_NAME,
        "--project", GCS_PROJECT_ID,
        "--check-shards",
        "--no-auth"
    ],
    capture_output=True,
    text=True
)

print(proc.stdout)
if proc.stderr:
    print(proc.stderr, file=sys.stderr)

if proc.returncode != 0:
    raise RuntimeError(
        f"Shards not found in GCS at gs://{pipeline_cfg.gcs_bucket_name}/{pipeline_cfg.gcs_cl_shards_prefix}\n"
        "Please run the pipeline to generate shards first, or upload pre-processed shards to GCS."
    )

print(f"\n✓ Shards verified - ready for streaming from GCS")
print(f"  Downstream cells will stream from: gs://{pipeline_cfg.gcs_bucket_name}/{pipeline_cfg.gcs_cl_shards_prefix}")

In [ ]:
# Cell 6: Dataset readiness probe — streams from GCS, no local download
# Uses src.dataset_probe (v2.5.11, 303 contract tests, frozen ProbeConfig).
# Polars can read directly from gs:// URIs for memory-efficient streaming.
import json
import os
import pathlib
import sys

os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

from src.dataset_probe import ProbeConfig, run_probe
from src.timer import cell_timer
from src.gcs_utils import get_gcs_path


def _get_cell6_logger():
    lg = logging.getLogger("cell6")
    lg.setLevel(logging.INFO)
    if not lg.handlers:
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(logging.Formatter("  %(message)s"))
        lg.addHandler(h)
    lg.propagate = False
    return lg


logger = _get_cell6_logger()

# Use GCS path for streaming (Polars supports gs:// URIs with gcsfs)
gcs_shard_path = get_gcs_path(pipeline_cfg.gcs_cl_shards_prefix, gcs_cfg)

out_path = pathlib.Path("logs/dataset_probe_report.json")
out_path.parent.mkdir(parents=True, exist_ok=True)

with cell_timer("Cell 6 - Dataset readiness probe (streaming from GCS)", logger=logger):
    logger.info("=" * 60)
    logger.info("  run_probe (streaming from GCS via Polars scan_ndjson)")
    logger.info("=" * 60)
    logger.info("  GCS path: %s" % gcs_shard_path)

    probe_cfg = ProbeConfig()
    
    report = run_probe(
        data_dir=gcs_shard_path,
        subset=1_465_484,
        full_scan=True,
        cfg=probe_cfg,
    )

    out_path.write_text(json.dumps(report.model_dump(), indent=2, default=str))
    logger.info("  report -> %s" % out_path)

    logger.info("\n" + "=" * 60)
    logger.info("  Probe summary")
    logger.info("=" * 60)
    summary = report.summary
    logger.info("  all_passed:     %s" % summary.get("all_passed"))
    logger.info("  passed_count:   %d" % summary.get("passed_count", 0))
    logger.info("  failed_block:   %d" % summary.get("failed_blocking_count", 0))
    logger.info("  subset_n:       %s" % format(report["subset_n"], ","))
    logger.info("  parse_errors:   %d" % report.get("parse_errors", 0))

    logger.info("\n  Gate results:")
    for gate_name, gate_result in report["gates"].items():
        status = "PASS" if gate_result.get("pass") else "FAIL"
        sev = gate_result.get("severity", "?")
        logger.info("    %-10s %s  (%s)" % (gate_name, status, sev))

    prov = report.get("provenance", {})
    logger.info("\n  probe_version: %s" % prov.get("probe_version"))
    logger.info("  polars_version: %s" % prov.get("polars_version"))
    logger.info("  full_scan: %s" % prov.get("full_scan"))
    logger.info("  streaming: GCS (no local download)")

    if not summary.get("all_passed"):
        raise RuntimeError("probe gates failed — see report")
    logger.info("\nOK all gates passed — corpus cleared for Stage 3")

In [ ]:
# Cell 7: Verify LePaRD dataset is present in GCS
import os
import subprocess
import sys

os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

print("=" * 60)
print("  Verifying LePaRD dataset in GCS")
print("=" * 60)

# Use the verification script
proc = subprocess.run(
    [
        ".venv/bin/python",
        "scripts/verify_gcs_data.py",
        "--bucket", GCS_BUCKET_NAME,
        "--project", GCS_PROJECT_ID,
        "--check-lepard",
        "--no-auth"
    ],
    capture_output=True,
    text=True
)

print(proc.stdout)
if proc.stderr:
    print(proc.stderr, file=sys.stderr)

if proc.returncode != 0:
    raise RuntimeError(
        f"LePaRD dataset not found in GCS at gs://{pipeline_cfg.gcs_bucket_name}/{pipeline_cfg.gcs_lepard_prefix}\n"
        "Please upload the LePaRD dataset to GCS first."
    )

from src.gcs_utils import get_gcs_path

lepard_gcs_path = get_gcs_path(
    pipeline_cfg.gcs_lepard_prefix + "lepard_train_4000000_rev0194f95.jsonl",
    gcs_cfg
)
print(f"\n✓ LePaRD verified - training will stream from: {lepard_gcs_path}")

In [ ]:
# Cell 8: LePaRD ↔ CourtListener compatibility audit
# Uses src.lepard_cl_compat (56 TDD tests) — pure analysis, deterministic,
# CI-gate capable. Reports usable gold pair rate (both endpoints in CL corpus).
import os
import pathlib
import subprocess
import sys

os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")


def _stream(cmd):
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        text=True,
        env=env,
    )
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    proc.wait()
    return proc.returncode


print("=" * 60)
print("  src.lepard_cl_compat — pair-level compatibility audit")
print("=" * 60)

rc = _stream([".venv/bin/python", "-m", "src.lepard_cl_compat"])
if rc != 0:
    raise SystemExit(f"compat audit failed with exit code {rc}")

print("\nOK compatibility audit complete")

In [ ]:
# Cell 9: Data quality gate — streams from GCS, no local download
# Uses scripts/audit_jsonl_nan.py — typed counters, W&B telemetry, CI gate.
# Audit script will stream shards directly from GCS using gs:// URIs.
import json
import os
import pathlib
import subprocess
import sys

os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

from src.gcs_utils import get_gcs_path

gcs_shard_path = get_gcs_path(pipeline_cfg.gcs_cl_shards_prefix, gcs_cfg)

print(f"Streaming shards from: {gcs_shard_path}")


def _stream(cmd):
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        text=True,
        env=env,
    )
    lines = []
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        lines.append(line)
    proc.wait()
    return proc.returncode, "".join(lines)


print("=" * 60)
print("  scripts/audit_jsonl_nan.py --json (streaming from GCS)")
print("=" * 60)

rc, output = _stream(
    [
        ".venv/bin/python",
        "scripts/audit_jsonl_nan.py",
        "--input-dir",
        gcs_shard_path,
        "--json",
    ]
)
if rc != 0:
    raise SystemExit(f"audit failed with exit code {rc}")

json_start = output.rfind("{")
if json_start >= 0:
    report = json.loads(output[json_start:])
    print("\n" + "=" * 60)
    print("  Audit summary (streamed from GCS)")
    print("=" * 60)
    print(f"  verdict:       {report.get('gate_verdict')}")
    print(f"  clean_pct:     {report.get('clean_pct')}")
    print(f"  total_lines:   {report.get('total_lines'):,}")
    print(f"  total_shards:  {report.get('total_shards')}")
    print(f"  nan_lines:     {report.get('nan_lines')}")
    print(f"  nonfinite:     {report.get('nonfinite_lines')}")
    print(f"  decode_errors: {report.get('decode_error_lines')}")

    if report.get("gate_verdict") != "CLEAN":
        raise RuntimeError(f"audit gate failed: {report.get('gate_verdict')}")

print("\nOK all shards pass data quality gate (streamed from GCS)")

In [ ]:
# Superseded: the upload step moved to Cell 5b (runs early, before the
# verify/probe/audit cells). This cell is a no-op kept as a placeholder so
# downstream cell numbering stays stable — please delete it manually if you
# want a cleaner notebook. To re-upload anything, re-run Cell 5b.
print("Upload step has moved to Cell 5b — nothing to do here.")
print(f"GCS bucket: gs://{GCS_BUCKET_NAME}/")
print(f"  cs1090b_cl_federal_appellate_bulk/  (shards)")
print(f"  cs1090b_lepard/                     (LePaRD)")
print(f"  cs1090b_cl_bulk/                    (CourtListener bulk CSVs)")


## Pipeline Summary — Stages 1–3 + Readiness Gates

End-to-end CourtListener + LePaRD data pipeline for the Legal RAG project. Every cell is thin orchestration over TDD-covered modules in `src/` and `scripts/` — no business logic lives in the notebook.

### Cell map

| Cell | Stage | Purpose | Modules / Scripts |
|---|---|---|---|
| **1** | — | Title and scope | (markdown) |
| **2** | Bootstrap | Clone repo, install `uv`, create `.venv` (Python 3.11.9), sync pinned deps, write `.env` | `uv`, `pyproject.toml`, `uv.lock` |
| **3** | Preflight | Reproducibility config + GPU/dependency contract + preflight gate | `src.repro`, `src.environment`, `src.timer` |
| **5** | GCS auth | Authenticate with GCS, load bucket/project from Colab secrets, build `PipelineConfig` + `GCSConfig` with `use_gcs_streaming=True` | `src.gcs_utils`, `src.config` |
| **5b** | Ensure in GCS | Idempotent upload: skip-if-in-GCS → download into Drive-backed staging → upload. Covers shards, LePaRD, and CourtListener bulk CSVs. Safe to re-run after Colab disconnects. | `scripts/upload_data_to_gcs.sh` → `src.bulk_download`, `src.s3_discovery`, `gsutil` |
| **6** | Verify bulk | Confirm the 4 CourtListener bulk CSVs are present in GCS | `scripts/verify_gcs_data.py` |
| **7** | (placeholder) | Legacy "skip bulk download" message — kept for cell-number stability | — |
| **8** | Verify shards | Confirm processed JSONL shards are present in GCS | `scripts/verify_gcs_data.py` |
| **9** | Stage 3 readiness | Full-corpus Polars `scan_ndjson` streamed from `gs://…` + 8 gates (schema, A7, A8, A9, A11, A12, A13, B6) | `src.dataset_probe` (v2.5.11, 303 contract tests) |
| **10** | Verify LePaRD | Confirm LePaRD training JSONL present in GCS | `scripts/verify_gcs_data.py` |
| **11** | Readiness | LePaRD ↔ CourtListener compatibility audit — reports usable gold pair rate | `src.lepard_cl_compat` (56 TDD tests) |
| **12** | Data quality | NaN / encoding / parse audit over all shards, streamed from GCS | `scripts/audit_jsonl_nan.py` |
| **13** | (placeholder) | Legacy "upload all" cell — superseded by Cell 5b, kept as a no-op | — |

### Persistence + idempotency strategy

* **Google Drive** is the single source of persistence for raw downloads.
  Cell 5b stages every CourtListener bulk CSV to
  `/content/drive/MyDrive/cs1090b_cl_bulk_staging/` before uploading to GCS,
  so a runtime disconnect mid-download simply resumes from the last fully
  completed file on Drive.
* **GCS** is the single source of truth for everything the downstream
  pipeline reads. Every verify cell checks `gs://…` via
  `scripts/verify_gcs_data.py`; every probe/audit/compat cell streams
  directly from `gs://…` URIs using `gcsfs` + Polars.
* **Idempotent fast-paths** — Cell 5b's script skips any file already in
  GCS (via `gsutil -q stat`) and any file already on Drive (via
  `src.bulk_download.download_file`'s existence check), so re-runs after a
  disconnect are cheap.

### Reproducibility guarantees

- **Python 3.11.9** pinned via `.python-version`
- **uv.lock** pins every dependency transitively (`torch 2.0.1+cu117`, `transformers 4.41.2`, `numpy 1.26.4`, …)
- **`src.repro.configure()`** sets `PYTHONHASHSEED=0`, `CUBLAS_WORKSPACE_CONFIG=:4096:8`, `torch.use_deterministic_algorithms(True)`, `cudnn.benchmark=False`, seeds Python/NumPy/torch CPU/CUDA RNGs to 0
- **LePaRD revision pinned** to `0194f95c3091acceab3b887c9b09ef432cf84052` (40-char SHA; mutable refs rejected)
- **Manifest checksums** — every shard's SHA-256 recorded; validation runs on every pipeline invocation
- **Contract tests** — `validate_pipeline` runs 13 `check_*` contract tests over sampled shards after extraction

### Downstream (not in this notebook)

README stages 4–7 — index generation (BM25 + FAISS), encoder training (BGE-M3 + Legal-BERT), sequential-loading evaluation (Tier A/B/C), W&B experiment tracking — are **not started** per the README and are correctly excluded from the data-pipeline notebook. They will be driven by `src.dataset_loader`, `src.lightning_datamodule`, `src.model_loader`, `src.split`, `src.wandb_logger`, and `src.hf_export` when that work begins.

### Verdict

> The full 1,465,484-record CourtListener Federal Appellate corpus plus the 4M-pair LePaRD training set are acquired, audited, probed, and verified compatible. The data pipeline is complete and reproducible across Colab sessions via Google Drive persistence. Ready for Stage 4 (indexing) whenever training begins.